# Main Experiment v1.4 — Prompt Wording and Scientific Conclusion Stability

**Frozen design:** 8 verified abstracts × 3 meaning-preserving prompt variants × 2 Qwen2.5 model sizes = **48 outputs**.

This notebook performs raw data collection only. It does not score or interpret outputs. Run all cells in order in Google Colab with a Tesla T4 GPU. The exact frozen dataset is distributed as `study_sources_v1_1_verified.json.gz`; Cell 2 decompresses it and verifies the original JSON SHA-256 before inference. Development/pilot outputs are excluded.


In [ ]:
# Cell 1 — Environment, imports, and GPU check
import os, json, csv, gc, time, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi
from google.colab import files

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 2 — Upload and validate the exact frozen dataset
import gzip
DATA_FILE = "study_sources_v1_1_verified.json.gz"
if not Path(DATA_FILE).exists():
    uploaded = files.upload()
    if DATA_FILE not in uploaded:
        raise FileNotFoundError(f"Upload {DATA_FILE} and rerun this cell.")
compressed_bytes = Path(DATA_FILE).read_bytes()
raw_bytes = gzip.decompress(compressed_bytes)
dataset_sha256 = hashlib.sha256(raw_bytes).hexdigest()
EXPECTED_DATASET_SHA256 = "8c2bc1018ee04dc216c55c389ba296c1ad8138566063b1add827ed06d8393295"
assert dataset_sha256 == EXPECTED_DATASET_SHA256, f"Dataset hash mismatch: {dataset_sha256}"
dataset = json.loads(raw_bytes.decode("utf-8"))
sources = dataset.get("sources")
assert dataset.get("dataset_version") == "v1.1"
assert isinstance(sources, list) and len(sources) == 8
required = {"id", "title", "url", "abstract", "doi", "license"}
assert len({s.get("id") for s in sources}) == 8, "Duplicate/missing source IDs"
for s in sources:
    assert required.issubset(s), f"Missing fields for {s.get('id')}"
    assert s["abstract"].strip() and s["url"].startswith("https://")
print("Dataset validation PASSED | version:", dataset["dataset_version"], "| sources:", len(sources), "| SHA-256:", dataset_sha256)
for s in sources:
    print(s["id"], "—", s["title"])


In [ ]:
# Cell 3 — Frozen v1.4 prompts, models, and generation settings
TASK_PARAPHRASES = {
    "P0_reference": "State the central conclusion of the scientific abstract below in exactly one sentence.",
    "P1_paraphrase": "In one sentence, report the main conclusion of the scientific abstract below.",
    "P2_paraphrase": "Write a single-sentence account of the principal conclusion drawn in the scientific abstract below.",
}
COMMON_INSTRUCTIONS = (
    " Use only information explicitly supported by the abstract; do not add external knowledge, "
    "invent quantities, or extend the source's scope; preserve important qualifications; "
    "return exactly one sentence and no additional commentary.\n\nABSTRACT:\n{abstract}"
)
MODELS = [
    {"model_id": "Qwen/Qwen2.5-0.5B-Instruct", "revision": "7ae557604adf67be50417f59c2c2f167def9a775"},
    {"model_id": "Qwen/Qwen2.5-1.5B-Instruct", "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"},
]
MAX_NEW_TOKENS = 100
DO_SAMPLE = False
SEED = 42
EXPECTED_OUTPUTS = len(sources) * len(TASK_PARAPHRASES) * len(MODELS)
assert EXPECTED_OUTPUTS == 48
print("Frozen design:", len(sources), "sources ×", len(TASK_PARAPHRASES), "prompts ×", len(MODELS), "models =", EXPECTED_OUTPUTS, "outputs")
for pid, phrase in TASK_PARAPHRASES.items(): print(pid, "→", phrase)


## Main run

The notebook uses the exact model revisions recorded in protocol v1.4. It processes one model at a time to conserve GPU memory and writes a checkpoint after every generated response. If the runtime disconnects, download/preserve the latest checkpoint before restarting. Do not combine runs from changed model revisions or software environments without documenting the change.


In [ ]:
# Cell 4 — Run all 48 outputs with per-output checkpointing
OUT_JSON = "main_experiment_raw_outputs_v1_4.json"
OUT_CSV = "main_experiment_raw_outputs_v1_4.csv"
results = []
run_started = datetime.now(timezone.utc).isoformat()
runtime = {
    "python": platform.python_version(), "pytorch": torch.__version__,
    "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0),
    "dataset_version": dataset.get("dataset_version"), "dataset_sha256": dataset_sha256,
    "protocol_version": "1.4", "run_started_utc": run_started,
}

def save_checkpoint():
    record = {
        "stage": "MAIN EXPERIMENT raw outputs; not scored",
        "protocol_version": "1.4",
        "runtime": runtime,
        "prompt_conditions": TASK_PARAPHRASES,
        "common_instructions": COMMON_INSTRUCTIONS,
        "generation_settings": {"do_sample": DO_SAMPLE, "max_new_tokens": MAX_NEW_TOKENS, "seed_before_each_generation": SEED},
        "records": results,
        "record_count": len(results),
        "expected_record_count": EXPECTED_OUTPUTS,
        "run_finished_or_checkpointed_utc": datetime.now(timezone.utc).isoformat(),
    }
    with open(OUT_JSON, "w", encoding="utf-8") as f: json.dump(record, f, ensure_ascii=False, indent=2)
    if results:
        with open(OUT_CSV, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=list(results[0].keys())); w.writeheader(); w.writerows(results)

for spec in MODELS:
    model_id, revision = spec["model_id"], spec["revision"]
    print("\nLoading", model_id, "at pinned revision", revision)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()
    for source in sources:
        for prompt_id, task_phrase in TASK_PARAPHRASES.items():
            exact_prompt = task_phrase + COMMON_INSTRUCTIONS.format(abstract=source["abstract"])
            chat_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": exact_prompt}], tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
            start = time.perf_counter()
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE,
                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
                )
            elapsed = time.perf_counter() - start
            n_input = inputs["input_ids"].shape[1]
            new_tokens = generated[0][n_input:]
            output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            row = {
                "source_id": source["id"], "source_title": source["title"], "source_url": source["url"],
                "source_abstract": source["abstract"], "model_id": model_id, "model_revision_sha": revision,
                "prompt_id": prompt_id, "task_phrase": task_phrase, "exact_prompt": exact_prompt,
                "raw_output": output_text, "input_tokens": int(n_input), "output_tokens": int(len(new_tokens)),
                "possible_truncation": bool(len(new_tokens) >= MAX_NEW_TOKENS),
                "generation_seconds": round(elapsed, 3), "run_time_utc": datetime.now(timezone.utc).isoformat(),
                "transformers_version": transformers.__version__, "pytorch_version": torch.__version__,
                "gpu": torch.cuda.get_device_name(0), "do_sample": DO_SAMPLE,
                "max_new_tokens": MAX_NEW_TOKENS, "seed_set_before_generation": SEED,
            }
            results.append(row); save_checkpoint()
            print(f"{len(results)}/{EXPECTED_OUTPUTS} | {source['id']} | {prompt_id} | {model_id.split('-')[-1]} | {row['output_tokens']} tokens")
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()

assert len(results) == EXPECTED_OUTPUTS, f"Expected {EXPECTED_OUTPUTS}, got {len(results)}"
assert len({(r['source_id'], r['prompt_id'], r['model_id']) for r in results}) == EXPECTED_OUTPUTS
save_checkpoint()
print("\nMAIN RAW COLLECTION COMPLETE:", len(results), "outputs")
print("Possible truncations:", sum(r["possible_truncation"] for r in results))
files.download(OUT_JSON)
files.download(OUT_CSV)


## Stop check and handoff

After the run, confirm that 48 unique source × prompt × model records exist and inspect possible truncations. Keep the JSON unchanged as raw evidence. Upload `main_experiment_raw_outputs_v1_4.json` to the assistant for source-grounded annotation and analysis. Do not treat the raw model outputs as findings until they have been evaluated against their original abstracts.
